In [1]:
import jax 
import jax.numpy as jnp
import jax.random as jrandom

from probjax.core import log_potential_fn, joint_sample
from probjax.core.custom_primitives.random_variable import rv 
from probjax.distributions import Normal


from probjax.inference.mcmc import MCMC 
from probjax.inference.transition_kernels import GaussianKernel
from probjax.inference.utils import flatten_fun

In [2]:
def f(key):
    key1, key2, key3 = jrandom.split(key, 3)
    z1 = rv(Normal(0, 1), "z1")(key1)
    z2  =z1 ** 2
    z3 = rv(Normal(z2, 1), "z2")(key2)
    z4 = z3 ** 3
    z5 = rv(Normal(z4, 1), "x")(key3)
    return z5

In [3]:
x = f(jrandom.PRNGKey(0))
sampler = joint_sample(f)
potential_fn = log_potential_fn(f, jrandom.PRNGKey(1))

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


In [4]:
init_samples =  sampler(jrandom.PRNGKey(0))
flat_args,  in_tree = jax.tree_util.tree_flatten(init_samples)

In [5]:
from jax._src.api_util import flatten_fun_nokwargs
from jax.linear_util import wrap_init

In [17]:
flat_potential_fn = flatten_fun(potential_fn, in_tree)

In [35]:
kernels = jax.tree_map(lambda x: GaussianKernel(float(x)), flat_args)
kernels

In [39]:
flat_kernels, in_tree_kernels = jax.tree_flatten(GaussianKernel(1.))

/tmp/ipykernel_1870/2731486154.py:1: FutureWarning: jax.tree_flatten is deprecated, and will be removed in a future release. Use jax.tree_util.tree_flatten instead.
  flat_kernels, in_tree_kernels = jax.tree_flatten(GaussianKernel(1.))


In [42]:
in_tree_kernels.num_leaves

1

In [36]:
mcmc = MCMC(kernels, potential_fn, ini)

In [37]:
mcmc.run(jrandom.PRNGKey(0),1000)

[Array(2.672, dtype=float32),
 Array(-0.193, dtype=float32),
 Array(1.053, dtype=float32)]